# Heatmap: Kongruenz nach Kanton und Akteur

Tabellarische **Heatmaps**: Wie stark Parole und kantonales Abstimmungsverhalten zusammenpassen – einmal **über alle Jahre** und **nach Zeitphase**.

**Was passiert hier?**
- Hilfsfunktionen für Skalierung und Parteinamen
- Statische Heatmap (Gesamtperiode)
- Interaktive Plotly-Heatmap mit Phasen-Slider
- Blog-HTML `d6_heatmap_zeitphasen.html` exportieren

**Datengrundlage**
- `data/processed/df_heatmap_with_positions.csv` (Gesamtperiode)
- `data/processed/df_heatmap_by_phase.csv` (pro Zeitphase, aus `2_berechnung.ipynb`)

**Vorher ausführen**
- `1_data_wrangling.ipynb` → `2_berechnung.ipynb`

**Danach**
- Blog: „Geografische Analyse“ (`d6_heatmap_zeitphasen.html`)
- Optional parallel: `3c_1_geomap.ipynb` (Kartenansicht derselben Daten)

## Setup


## Import und Hilfsfunktionen


Heatmap-Funktionen aus `visualisierungen` laden.


In [ ]:
# autoreload: Änderungen an visualisierungen.py ohne Kernel-Neustart
%load_ext autoreload
%autoreload 2

import importlib
import re
import pandas as pd
import visualisierungen

importlib.reload(visualisierungen)

from visualisierungen import (
    heatmap,
    heatmap_interaktiv_phasen,
    write_plotly_html_responsive,
)


Gemeinsame Hilfen: GLP ausschliessen, feste Akteurs-Reihenfolge, Skala −0.5 … +0.5.


In [ ]:
# GLP ausschliessen – zu wenig Abstimmungen um aussagekräftige Werte zu berechnen
PARTEI_EXCLUDE = {"p-glp_label"}

# Feste Reihenfolge der Akteure in der Heatmap (von oben nach unten)
PARTEI_ORDER = [
    "br-pos_label",
    "bv-pos_label",
    "p-gps_label",
    "p-sps_label",
    "p-mitte_label",
    "p-fdp_label",
    "p-svp_label",
]


def prepare_heatmap_df(df):
    # Index-Spalte entfernen falls vorhanden (entsteht beim CSV-Export ohne index=False)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    # GLP herausfiltern und partei-Spalte als Zeilenindex setzen
    if "partei" in df.columns:
        df = df[~df["partei"].isin(PARTEI_EXCLUDE)].set_index("partei")
    else:
        df = df[~df.index.isin(PARTEI_EXCLUDE)]
    # Zeilen in die gewünschte Reihenfolge bringen – nur vorhandene Akteure übernehmen
    order = [p for p in PARTEI_ORDER if p in df.index]
    return df.loc[order]


def to_kongruenz_scale(M):
    # Metaspalten entfernen – nur Kantone sollen als Spalten übrig bleiben
    M = M.drop(columns=["jahr", "phase", "abstimmung"], errors="ignore").astype(float)
    # Skalierung normalisieren: Werte aus 2_berechnung kommen als Prozentpunkte (0-100),
    # der Plot erwartet −0.5 bis +0.5 → durch 100 teilen
    if M.abs().max().max() > 1:
        M = M / 100
    return M.clip(-0.5, 0.5)


def party_short(name):
    # Rohe Spaltennamen wie "p-sps_label" in lesbare Kürzel ("SP") umwandeln
    s = str(name)
    if "-" not in s or "_" not in s:
        return s
    pre, rest = s.split("-", 1)
    mid = rest.split("_", 1)[0]
    if mid == "pos":
        if pre == "br":
            return "Bundesrat"
        if pre == "bv":
            return "Bundesver."
    if pre == "p":
        return {"gps": "Grüne", "sps": "SP", "mitte": "Mitte", "fdp": "FDP", "svp": "SVP"}.get(mid, mid.upper())
    return f"{pre.upper()}-{mid.upper()}"

## Heatmap: Gesamtperiode

Alle Jahre zusammen – Kantone in Spalten, Akteure in Zeilen.


Kantons-Kongruenz über alle Jahre laden.


In [ ]:
# Gesamtperiode laden – alle Jahre zusammengefasst, ein Wert pro Kanton und Akteur
df = pd.read_csv("../data/processed/df_heatmap_with_positions.csv")
df.head()

Daten für die Gesamt-Heatmap aufbereiten.


In [ ]:
# GLP entfernen, Reihenfolge festlegen und Werte auf −0.5 bis +0.5 skalieren
df_plot = to_kongruenz_scale(prepare_heatmap_df(df))
df_plot

Statische Heatmap: Kantone × Akteure (Matplotlib).


In [ ]:
# Kantonsabkürzungen aus den Spaltennamen ableiten (z.B. "zh" → "ZH")
cantons = [str(c)[:2].upper() for c in df_plot.columns]
# Lesbare Akteursnamen aus den Zeilenindizes ableiten (z.B. "p-sps_label" → "SP")
parties = [party_short(i) for i in df_plot.index]

heatmap(
    df_plot,
    xlabel=None,
    ylabel=None,
    xlabels=cantons,
    ylabels=parties,
    figsize=(16, 6),
)

## Heatmap: nach Zeitphase

Gleiche Logik, aber mit Phasen-Slider (interaktiv für den Blog).


Heatmap-Daten pro historischer Phase laden.


In [ ]:
# Phasendaten laden – gleiche Struktur wie Gesamtperiode, aber mit zusätzlicher "phase"-Spalte
df_phase = pd.read_csv("../data/processed/df_heatmap_by_phase.csv")
# Überflüssige Index-Spalte entfernen – entsteht beim CSV-Export ohne index=False
if "Unnamed: 0" in df_phase.columns:
    df_phase = df_phase.drop(columns=["Unnamed: 0"])
# GLP bereits hier herausfiltern damit alle nachfolgenden Schritte sauber bleiben
df_phase = df_phase[~df_phase["partei"].isin(PARTEI_EXCLUDE)]
df_phase.head()

Anzeigenamen der fünf Zeitphasen (Slugs aus `2_berechnung`).


In [ ]:
# Lesbare Phasentitel – Slugs kommen aus df_phase, Titel müssen mit pd.cut in 2_berechnung übereinstimmen
PHASE_TITLES = {
    "phase1_fruehphase": "Frühphase\n(1848-1899)",
    "phase2_volatile": "Volatile Phase\n(1900-1949)",
    "phase3_konsens": "Konsensphase\n(1950-1975)",
    "phase4_aufspaltung": "Aufspaltung\n(1976-2009)",
    "phase5_2010_heute": "2010er–heute\n(2010-heute)",
}


def _phase_order(slug: str) -> int:
    # Phasen anhand der eingebetteten Zahl sortieren statt alphabetisch –
    # ohne diese Funktion würde z.B. phase5 vor phase1 landen
    m = re.search(r"phase(\d+)", str(slug))
    return int(m.group(1)) if m else 99


# Alle vorhandenen Phasen aus den Daten holen und chronologisch sortieren
phase_slugs = sorted(df_phase["phase"].dropna().unique(), key=_phase_order)
PHASES = [(slug, PHASE_TITLES.get(slug, slug)) for slug in phase_slugs]

# Final Plot D6: Interaktive Heatmap bauen und für den Blog als HTML speichern.

In [ ]:
# Für jede Phase einen aufbereiteten DataFrame bauen und in der Liste sammeln
phasen_plot = []
for slug, titel in PHASES:
    sub = df_phase.loc[df_phase["phase"] == slug]
    if sub.empty:
        print(f"Übersprungen (keine Daten): {titel}")
        continue
    # Phasenspalte entfernen bevor die Skalierung läuft – sie enthält Text, keine Zahlen
    df_plot = to_kongruenz_scale(
        prepare_heatmap_df(sub.drop(columns=["phase"], errors="ignore"))
    )
    phasen_plot.append((titel, df_plot))

if phasen_plot:
    # Kantonsabkürzungen und Akteursnamen aus der ersten Phase ableiten – alle Phasen haben dieselbe Struktur
    cantons = [str(c)[:2].upper() for c in phasen_plot[0][1].columns]
    parties = [party_short(i) for i in phasen_plot[0][1].index]

    # Interaktive Heatmap mit Phasen-Slider erstellen
    fig_phase = visualisierungen.heatmap_interaktiv_phasen(
        phasen_plot,
        xlabel=None,
        ylabel=None,
        xlabels=cantons,
        ylabels=parties,
        height=360,
    )
    fig_phase.show()

    # Als responsive HTML exportieren für die Einbettung im Blog
    visualisierungen.write_plotly_html_responsive(
        fig_phase,
        "../Blog/blog_plots/d6_heatmap_zeitphasen.html",
        height=400,
        phase_bar_labels=[titel for titel, _ in phasen_plot],
    )
else:
    print("Keine Phasen-Daten für Heatmap.")